In [1]:
import pathlib
import cogent3
import madb 
from typing import Dict, List
import dataclasses
import shutil
from thesis_rec import create_thesis_rec, thesis_rec, thesis_rec_from_alignment
from time import sleep
from tqdm import tqdm

source_alignment_path = pathlib.Path('~/source/ensembl/primates100')
thesis_data = pathlib.Path('~/source/ensembl/thesis_data')
max_workers = 4

primates = {'gorilla': 'gorilla_gorilla', 'chimp': 'pan_troglodytes', 'macaque': 'macaca_mulatta'}


# Extract pairs of primates from alignments of all primates
 - rename sequences names to species name 
 - select only alignments that contain all 4 species 
 - select only the specific pair of species
 - remove common gaps

In [2]:

@cogent3.app.composable.define_app
def rename(align: cogent3.app.typing.AlignedSeqsType)->cogent3.app.typing.AlignedSeqsType:
    sleep(1)
    return align.rename_seqs(lambda x: x.split(':')[0])

with tqdm(total=len(primates), desc="Generating primate pair alignments", unit="step") as pbar:
    for primate in primates:
        pair_name = f'human_{primate}'
        pair_path_root = thesis_data / pair_name
        pbar.set_description(f"Generating {pair_name} pair alignments")

        in_dstore = cogent3.open_data_store(source_alignment_path, suffix='fa') 
        out_dstore = cogent3.open_data_store(pair_path_root/'ensembl_alignments', suffix='fa', mode='w') 

        loader = cogent3.get_app('load_aligned', moltype='dna')
        select_4_primates = cogent3.get_app('take_named_seqs','homo_sapiens',*primates.values())
        select_pair = cogent3.get_app('take_named_seqs','homo_sapiens',primates[primate])
        omit_gaps = cogent3.get_app('omit_gap_pos', moltype="dna")
        writer = cogent3.get_app('write_seqs', data_store = out_dstore)
        app = loader + rename() + select_4_primates + select_pair + omit_gaps + writer 
        app.apply_to(in_dstore, show_progress=True, parallel=True, par_kw=dict(max_workers=max_workers))
        pbar.update(1)
        sleep(10)


Generating human_gorilla pair alignments:   0%|          | 0/3 [00:00<?, ?step/s]

   0%|          |00:00<?

Generating human_chimp pair alignments:  33%|███▎      | 1/3 [00:14<00:09,  4.54s/step]  

   0%|          |00:00<?

Generating human_macaque pair alignments:  67%|██████▋   | 2/3 [00:29<00:10, 10.41s/step]

   0%|          |00:00<?

Generating human_macaque pair alignments: 100%|██████████| 3/3 [00:43<00:00, 14.54s/step]


In [3]:
@cogent3.app.composable.define_app
def rec_from_alignment(aln: cogent3.app.typing.AlignedSeqsType)->cogent3.app.typing.SerialisableType:
    sleep(1)
    return thesis_rec_from_alignment(aln)

@cogent3.app.composable.define_app
def cogent3_alignment(rec: cogent3.app.typing.SerialisableType)->cogent3.app.typing.SerialisableType:
    rec = thesis_rec.from_rich_dict(rec)
    sleep(1)
    return rec.align_cogent3()

with tqdm(total=len(primates), desc="Generating cogent3 alignments", unit="step") as pbar:
    for primate in primates:
        pair_name = f'human_{primate}'
        pair_path_root = thesis_data / pair_name
        pbar.set_description(f"Generating {pair_name} cogent3 alignments")

        in_dstore = cogent3.open_data_store(pair_path_root / 'ensembl_alignments', suffix='fa') 
        out_dstore = cogent3.open_data_store(pair_path_root / 'cogent3', suffix='json', mode='w') 

        loader = cogent3.get_app('load_aligned', moltype='dna')
        writer = cogent3.get_app('write_json', data_store=out_dstore)
        app = loader + rec_from_alignment() + cogent3_alignment() + writer 
        app.apply_to(in_dstore, show_progress=True, parallel=True, par_kw=dict(max_workers=max_workers))
        if len(out_dstore.not_completed) > 0:
            print(cogent3.util.deserialise.deserialise_object(out_dstore.not_completed[0].read()))
        pbar.update(1)
        sleep(10)


   0%|          |00:00<?

NotCompleted(type=ERROR, origin=rec_from_alignment, source="ENSG00000162482.fa", message="Traceback (most recent call last):
  File "/home/richard/source/ensembl/.venv/lib/python3.12/site-packages/cogent3/app/composable.py", line 401, in _call
    result = self.main(val, *args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/richard/source/ensembl/.venv/lib/python3.12/site-packages/cogent3/app/composable.py", line 461, in _main
    return self._user_func(**bound.arguments)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_3609479/24052152.py", line 4, in rec_from_alignment
  File "/home/richard/source/ensembl/thesis_rec.py", line 211, in thesis_rec_from_alignment
    raise ValueError("Alignment must contain exactly two sequences.")
ValueError: Alignment must contain exactly two sequences.
")


Generating human_macaque cogent3 alignments: 100%|██████████| 3/3 [00:32<00:00, 10.77s/step]


In [5]:
@cogent3.app.composable.define_app
def sw(rec: cogent3.app.typing.SerialisableType, timeout_sec: int = 600)->cogent3.app.typing.SerialisableType:
    rec = thesis_rec.from_rich_dict(rec)
    print(f'calculating smith_waterman on {rec.unique_id}')
    sleep(1)
    if '' in rec.unaligned_seqs.values():
        return cogent3.NotCompleted("Empty sequence")
    if rec.unique_id == 'ENSG00000143199.fa':
        return cogent3.NotCompleted("skipping ENSG00000143199.fa")
    return rec.calc_sw(timeout_sec).to_rich_dict()

with tqdm(total=len(primates), desc="Generating ungapped smith-waterman", unit="step") as pbar:
    # primate = 'chimp'
    for primate in primates:
        pair_name = 'human_'+primate
        pair_path_root = thesis_data / pair_name
        pbar.set_description(f"Generating {pair_name} cogent3 alignments")

        in_dstore = cogent3.open_data_store(pair_path_root/'cogent3', suffix='json') 
        out_dstore = cogent3.open_data_store(pair_path_root/'smith_waterman', suffix='json', mode='w') 

        loader = cogent3.get_app('load_json')
        writer = cogent3.get_app('write_json', data_store = out_dstore)
        app = loader + sw(300) + writer 
        app.apply_to(in_dstore, show_progress=True)
        pbar.update(1)
        sleep(10)

calculating smith_waterman on ENSG00000143199.fa


   0%|          |00:00<?

calculating smith_waterman on ENSG00000198691-1.fa


/home/richard/source/ensembl/.venv/lib/python3.12/site-packages/cogent3/align/pairwise.py:1095: UserWarning: Local alignment will use > 68731.19919Mb.
  self.results[dp_options] = self.emission_probs.dp(
/home/richard/source/ensembl/thesis_rec.py:170: UserWarning: SW alignment failed for ENSG00000198691-1.fa: 'NotCompleted' object has no attribute 'seqs'
  warnings.warn(f"SW alignment failed for {self.unique_id}: {result}")


calculating smith_waterman on ENSG00000117020-1.fa


...

/home/richard/source/ensembl/.venv/lib/python3.12/site-packages/cogent3/align/pairwise.py:1095: UserWarning: Local alignment will use > 388974.34653Mb.
  self.results[dp_options] = self.emission_probs.dp(
/home/richard/source/ensembl/thesis_rec.py:170: UserWarning: SW alignment failed for ENSG00000154027-1.fa: 'NotCompleted' object has no attribute 'seqs'
  warnings.warn(f"SW alignment failed for {self.unique_id}: {result}")


calculating smith_waterman on ENSG00000117020-0.fa


/home/richard/source/ensembl/.venv/lib/python3.12/site-packages/cogent3/align/pairwise.py:1095: UserWarning: Local alignment will use > 390949.04535Mb.
  self.results[dp_options] = self.emission_probs.dp(
/home/richard/source/ensembl/thesis_rec.py:170: UserWarning: SW alignment failed for ENSG00000117020-0.fa: 'NotCompleted' object has no attribute 'seqs'
  warnings.warn(f"SW alignment failed for {self.unique_id}: {result}")


calculating smith_waterman on ENSG00000162618.fa


/home/richard/source/ensembl/.venv/lib/python3.12/site-packages/cogent3/align/pairwise.py:1095: UserWarning: Local alignment will use > 768285.9903Mb.
  self.results[dp_options] = self.emission_probs.dp(
/home/richard/source/ensembl/thesis_rec.py:170: UserWarning: SW alignment failed for ENSG00000162618.fa: 'NotCompleted' object has no attribute 'seqs'
  warnings.warn(f"SW alignment failed for {self.unique_id}: {result}")
Generating human_macaque cogent3 alignments: 100%|██████████| 3/3 [23:12<00:00, 464.15s/step]


In [ ]:
@cogent3.app.composable.define_app
def align_madb(rec: cogent3.app.typing.SerialisableType)->cogent3.app.typing.SerialisableType:
    rec = thesis_rec.from_rich_dict(rec)
    print(f'aligning via madb {rec.unique_id}')
    sleep(1)
    result = rec.align_madb(120).to_rich_dict()
    return result

with tqdm(total=len(primates), desc="Generating ungapped smith-waterman", unit="step") as pbar:
    # primate = 'chimp'
    # for primate in primates:
    primate = 'macaque'
    pair_name = f'human_{primate}'
    pair_path_root = thesis_data / pair_name
    pbar.set_description(f"Generating {pair_name} cogent3 alignments")

    in_dstore = cogent3.open_data_store(pair_path_root/'smith_waterman', suffix='json')
    out_dstore = cogent3.open_data_store(pair_path_root/'madb', suffix='json', mode='w') 

    loader = cogent3.get_app('load_json')
    writer = cogent3.get_app('write_json', data_store = out_dstore)
    app = loader + align_madb() + writer
    app.apply_to(in_dstore, show_progress=True)
    print(out_dstore.summary_logs)
    pbar.update(1)
    sleep(10)

aligning via madb ENSG00000126070-0.fa




/home/richard/source/ensembl/thesis_rec.py:166: UserWarning: MADB alignment for ENSG00000126070-0.fa, k=10 failed: Input sequences are not all the same length: {1037, 2085}. Please ensure all sequences are properly aligned or correct the input file format.
  warnings.warn(f"MADB alignment for {self.unique_id}, k={k} failed: {result}")



































Aligning madb (k-mer size: 95): 100%|██████████| 18/18 [00:31<00:00,  1.73s/step]


   0%|          |00:00<?

aligning via madb ENSG00000134249-0.fa




/home/richard/source/ensembl/thesis_rec.py:166: UserWarning: MADB alignment for ENSG00000134249-0.fa, k=10 failed: Input sequences are not all the same length: {1493, 2790}. Please ensure all sequences are properly aligned or correct the input file format.
  warnings.warn(f"MADB alignment for {self.unique_id}, k={k} failed: {result}")
















/home/richard/source/ensembl/thesis_rec.py:166: UserWarning: MADB alignment for ENSG00000134249-0.fa, k=50 failed: Input sequences are not all the same length: {1472, 1469}. Please ensure all sequences are properly aligned or correct the input file format.
  warnings.warn(f"MADB alignment for {self.unique_id}, k={k} failed: {result}")


/home/richard/source/ensembl/thesis_rec.py:166: UserWarning: MADB alignment for ENSG00000134249-0.fa, k=55 failed: Input sequences are not all the same length: {1472, 1469}. Please ensure all sequences are properly aligned or correct the input file format.
  warnings.warn(f"MADB alignment for {self.unique

...

aligning via madb ENSG00000184389.fa






/home/richard/source/ensembl/thesis_rec.py:166: UserWarning: MADB alignment for ENSG00000184389.fa, k=15 failed: Input sequences are not all the same length: {24794, 42298}. Please ensure all sequences are properly aligned or correct the input file format.
  warnings.warn(f"MADB alignment for {self.unique_id}, k={k} failed: {result}")


/home/richard/source/ensembl/thesis_rec.py:166: UserWarning: MADB alignment for ENSG00000184389.fa, k=20 failed: Input sequences are not all the same length: {16329, 15991}. Please ensure all sequences are properly aligned or correct the input file format.
  warnings.warn(f"MADB alignment for {self.unique_id}, k={k} failed: {result}")


/home/richard/source/ensembl/thesis_rec.py:166: UserWarning: MADB alignment for ENSG00000184389.fa, k=25 failed: Input sequences are not all the same length: {14146, 14463}. Please ensure all sequences are properly aligned or correct the input file format.
  warnings.warn(f"MADB alignment for {self.unique_id}, k={k} 

aligning via madb ENSG00000116771.fa




/home/richard/source/ensembl/thesis_rec.py:155: UserWarning: MADB alignment for ENSG00000116771.fa, k=10 timed out after 120s
  warnings.warn(f"MADB alignment for {self.unique_id}, k={k} timed out after {timeout_sec}s")


/home/richard/source/ensembl/thesis_rec.py:155: UserWarning: MADB alignment for ENSG00000116771.fa, k=15 timed out after 120s
  warnings.warn(f"MADB alignment for {self.unique_id}, k={k} timed out after {timeout_sec}s")


/home/richard/source/ensembl/thesis_rec.py:155: UserWarning: MADB alignment for ENSG00000116771.fa, k=20 timed out after 120s
  warnings.warn(f"MADB alignment for {self.unique_id}, k={k} timed out after {timeout_sec}s")


/home/richard/source/ensembl/thesis_rec.py:155: UserWarning: MADB alignment for ENSG00000116771.fa, k=25 timed out after 120s
  warnings.warn(f"MADB alignment for {self.unique_id}, k={k} timed out after {timeout_sec}s")


/home/richard/source/ensembl/thesis_rec.py:155: UserWarning: MADB alignment for ENSG00000116771.fa, k=30 timed

aligning via madb ENSG00000143382.fa
